# Notebook 12 — OpenADMET MA-ST-RAE scoring and blind-test submission

**Research goal:** Evaluate the actual Notebook 11 nested out-of-fold predictions under the challenge's **soft-threshold relative absolute error** (ST-RAE), inspect all four CYP endpoints and bootstrap uncertainty, and generate a **750-compound blind-test direct-inhibition regression submission** from the selected CheMeleon + screening-transfer ensemble.

> **Key constraint:** `cyp-challenge-TEST-BLINDED.csv` has only molecule identifiers/structures. **No ground-truth pIC50 or credible intervals are available. It is mathematically impossible to score that blind set locally.** The notebook evaluates already-held-out *labeled* Notebook 11 predictions, then refits the selected models on the public training set and generates an **unscored** test submission. Only the challenge organizer can report the blind-test MA-ST-RAE.

### Methodological contract

* Endpoint score = mean distance **outside experimental credible intervals** divided by the same distance from the constant predictor equal to the evaluated group's mean label; the four endpoint ratios are macro-averaged. 1,000 paired compound bootstrap resamples estimate the scoring distribution.
* The score is recomputed on every bootstrap sample, **including its baseline denominator**. The local OOF estimate and bootstrap estimator are not necessarily numerically identical.
* All Notebook 11 *weight choices* are retained (50% CheMeleon, 50% ScreenStack). They are **not reoptimized using test labels**.
* The test predictions use **only structural features** at inference. Measured test screening values are neither required nor accessed.
* Outputs are labeled `local_oof` or `blind_unscored`. A synthetic/proxy smoke run is **never** labeled an official submission.

**Sources:** [challenge tutorial](https://github.com/OpenADMET/CYP-Challenge-Tutorial), [public train/test dataset](https://huggingface.co/datasets/openadmet/cyp-challenge-train-test), [OpenADMET scoring framework](https://github.com/OpenADMET/blind-challenge-template). Challenge closes **November 3, 2026 at 23:59 UTC** (check platform for the authoritative deadline).

## 0 · Install dependencies

Run the following in Google Colab; a T4 GPU is recommended when `RUN_BLIND_REFIT=True`. The diagnostic OOF scoring portion works on CPU.

In [ ]:
%pip -q install "rdkit>=2023.9" "lightgbm>=4.3" "scikit-learn>=1.6" "scipy>=1.11" "pandas>=2" "matplotlib>=3.7" "chemprop>=2.1" "lightning>=2.3" "torch>=2.3"

## 1 · User settings

Upload `cyp_notebook_11_results.zip` to the Colab session (or set its Google Drive path). For a real submission, **keep** `FAST_MODE=False`, `GRAPH_PROXY_SMOKE_ONLY=False`, `RUN_BLIND_REFIT=True`. All caches live under `PERSISTENT_ROOT`, if configured.

Set `DATA_OVERRIDES` only when using manually uploaded CSVs or controlled synthetic tests. The default uses official public Hugging Face CSVs.

In [ ]:
from pathlib import Path
from urllib.request import urlopen, urlretrieve
import os, io, sys, json, time, zipfile, hashlib, random, warnings, platform, importlib.util, inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
import lightgbm as lgb
from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold

SEED = 2026
ISOFORMS = ['CYP1A2', 'CYP2C9', 'CYP2D6', 'CYP3A4']
NB11_RESULTS_ZIP = 'cyp_notebook_11_results.zip'  # Colab upload or absolute Drive path
PERSISTENT_ROOT = None  # e.g., '/content/drive/MyDrive/CYP/Notebook12'
RUN_BLIND_REFIT = True
FAST_MODE = False  # True is ONLY a smoke test; never submit fast-mode outputs
GRAPH_PROXY_SMOKE_ONLY = False  # True is ONLY for synthetic/offline testing
RUN_GRAPH = True
N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 0
FINGERPRINT_BITS = 2048
SCREEN_CROSSFIT_FOLDS = 2 if FAST_MODE else 3
N_LGBM_ESTIMATORS = 65 if FAST_MODE else 350
N_SCREEN_ESTIMATORS = 55 if FAST_MODE else 180
MAX_GRAPH_EPOCHS = 3 if FAST_MODE else 22
GRAPH_PATIENCE = 2 if FAST_MODE else 5
GRAPH_BATCH_SIZE = 64
DATA_OVERRIDES = {}  # {'direct':'/content/direct.csv','single':'/content/screen.csv','test':'/content/test.csv'}
EXPERIMENT_TAG = 'official_score_and_blind_refit_v1'
ROOT_URL = 'https://huggingface.co/datasets/openadmet/cyp-challenge-train-test/resolve/main/'
FILES = {'direct':'cyp-challenge-TRAIN_inhibition.csv',
         'single':'cyp-challenge-single-concentration-TRAIN.csv',
         'test':'cyp-challenge-TEST-BLINDED.csv'}
BASE = Path(PERSISTENT_ROOT) if PERSISTENT_ROOT else Path('.')
OUT = BASE / 'cyp_notebook_12_results'; OUT.mkdir(parents=True,exist_ok=True)
CACHE = BASE / 'cyp_notebook_12_cache'; CACHE.mkdir(parents=True,exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
print('Python',platform.python_version(),'RDKit',rdBase.rdkitVersion,'LightGBM',lgb.__version__)
print('Output folder:',OUT.resolve())
if FAST_MODE or GRAPH_PROXY_SMOKE_ONLY:
    print('SMOKE/PROXY MODE: generated predictions are NOT valid competition submissions')

## 2 · Reference-compatible ST-RAE implementation and self-checks

For one enzyme, the numerator is the **sum of distances outside the 95% credible interval**. The denominator uses the same interval-distance loss, but predicts the **mean of the ground-truth pIC50s in the evaluated sample** for every compound. Macro MA-ST-RAE is the simple mean of the four ratios. The implementation is intentionally strict: missing or reversed intervals raise errors instead of silently replacing ST-RAE with ordinary RAE.

To provide a defensible reproducibility check, the next cell also attempts to download the tutorial's `evaluation/custom_scoring_functions.py` and compares its `rae_soft_threshold_absolute_error` on a toy cohort. If the public scorer is unreachable or its API changes, that comparison will be reported as **not verified**, not as a successful test.

In [ ]:
def soft_interval_distance(y_pred, ci_low, ci_high):
    pred, lo, hi = (np.asarray(x, dtype=np.float64) for x in (y_pred,ci_low,ci_high))
    return np.maximum(lo-pred,0.0) + np.maximum(pred-hi,0.0)

def st_rae(y_true, y_pred, ci_low, ci_high):
    truth, pred, lo, hi = (np.asarray(x, dtype=np.float64).ravel() for x in
                            (y_true,y_pred,ci_low,ci_high))
    if not (len(truth)==len(pred)==len(lo)==len(hi) and len(truth)>1):
        raise ValueError('ST-RAE needs at least two aligned labeled measurements.')
    if not (np.isfinite(truth).all() and np.isfinite(pred).all()
            and np.isfinite(lo).all() and np.isfinite(hi).all() and np.all(lo<=hi)):
        raise ValueError('Missing/nonfinite true values, predictions or credible intervals.')
    num=float(np.sum(soft_interval_distance(pred,lo,hi)))
    anchor=float(truth.mean())
    denom=float(np.sum(soft_interval_distance(np.full_like(truth,anchor),lo,hi)))
    if denom<=1e-14:
        raise ZeroDivisionError('ST-RAE denominator is zero: evaluated cohort is degenerate.')
    return num/denom

def macro_st_rae(by_isoform):
    assert set(by_isoform)==set(ISOFORMS), 'All four CYP endpoints must be present.'
    return float(np.mean([st_rae(*by_isoform[iso]) for iso in ISOFORMS]))

# Tests: zero loss inside CI; edge distance; ensemble of endpoints, bad bounds fail closed.
assert np.allclose(soft_interval_distance([4,5,7],[3,5,6],[5,5.5,6.5]),[0,.0,.5])
test_truth=np.array([3.,4.,6.,7.]); bounds_lo=test_truth-.1; bounds_hi=test_truth+.1
assert abs(st_rae(test_truth,test_truth,bounds_lo,bounds_hi))<1e-12
assert abs(st_rae(test_truth,np.repeat(test_truth.mean(),4),bounds_lo,bounds_hi)-1)<1e-12
try:
    st_rae(test_truth,test_truth,bounds_lo,np.array([np.nan,4.,6.,7.]))
    raise AssertionError('Nonfinite CI was not rejected')
except ValueError: pass
print('Local ST-RAE behavior checks passed.')

In [ ]:
OFFICIAL_PARITY = {'status':'not_checked','details':''}
reference_url = ('https://raw.githubusercontent.com/OpenADMET/CYP-Challenge-Tutorial/'
                 'main/evaluation/custom_scoring_functions.py')
try:
    reference_file = CACHE/'openadmet_custom_scoring_functions.py'
    if not reference_file.exists():
        with urlopen(reference_url,timeout=25) as fh:
            reference_file.write_bytes(fh.read())
    spec=importlib.util.spec_from_file_location('openadmet_reference_scoring',reference_file)
    reference=importlib.util.module_from_spec(spec);spec.loader.exec_module(reference)
    fn=getattr(reference,'rae_soft_threshold_absolute_error')
    y=np.array([3.0,4.0,5.0,7.0,8.0])
    p=np.array([2.5,4.2,6.0,6.6,7.1])
    lo=y-0.2; hi=y+0.3
    sig=inspect.signature(fn)
    params=sig.parameters
    if 'y_true_lower' in params and 'y_true_upper' in params:
        reference_value=fn(y,p,y_true_lower=lo,y_true_upper=hi)
    elif 'y_true_low' in params and 'y_true_high' in params:
        reference_value=fn(y,p,y_true_low=lo,y_true_high=hi)
    else:
        reference_value=fn(y,p,lo,hi)
    ours=st_rae(y,p,lo,hi)
    assert np.isclose(float(reference_value),ours,rtol=1e-10,atol=1e-10),(
        f'Metric disagreement: official={reference_value}, local={ours}')
    OFFICIAL_PARITY={'status':'passed','details':f'Official function {sig} matched toy cohort',
        'reference_sha256':hashlib.sha256(reference_file.read_bytes()).hexdigest()}
except Exception as exc:
    OFFICIAL_PARITY={'status':'unverified','details':repr(exc)}
print('Official source parity:',OFFICIAL_PARITY['status'], OFFICIAL_PARITY['details'])
if OFFICIAL_PARITY['status'] != 'passed':
    print('WARNING: local implementation is source-formula compatible but direct official-code parity was not confirmed.')

## 3 · Load Notebook 11 OOF predictions and validate coverage

Scores are calculated **only on the original labeled held-out predictions**. No model training occurs in this section. The input archive must contain the real five-fold `nested_outer_predictions.csv` and `run_metadata.json` from Notebook 11; proxy/synthetic results are rejected for scientific scoring.

In [ ]:
def locate_archive(filename):
    p=Path(filename)
    if p.is_file():return p
    for root in [Path('/content'),Path('/mnt/data'),BASE]:
        candidate=root/p.name
        if candidate.is_file():return candidate
    raise FileNotFoundError(
        f'Cannot locate {filename}. Upload cyp_notebook_11_results.zip in Colab or set NB11_RESULTS_ZIP.')

def zip_csv(z, suffix):
    names=[name for name in z.namelist() if name.endswith('/'+suffix) or name==suffix]
    if len(names)!=1:raise ValueError(f'Expected one {suffix} in ZIP, got {names}')
    return pd.read_csv(io.BytesIO(z.read(names[0])))

def zip_json(z, suffix):
    names=[name for name in z.namelist() if name.endswith('/'+suffix) or name==suffix]
    if len(names)!=1:raise ValueError(f'Expected one {suffix} in ZIP, got {names}')
    return json.loads(z.read(names[0]))

nb11_path=locate_archive(NB11_RESULTS_ZIP)
with zipfile.ZipFile(nb11_path) as archive:
    preds=zip_csv(archive,'nested_outer_predictions.csv')
    info11=zip_json(archive,'run_metadata.json')
    manifest11=zip_csv(archive,'fold_manifest.csv')
print('Notebook11:',nb11_path,'total predictions',len(preds))
assert info11.get('complete_five_fold_oof') and info11.get('scientifically_valid_full_model_run'), (
    'Notebook11 archive is incomplete/fast/proxy; cannot report research-grade OOF scores')
required=['fold','subset','canonical_smiles','scaffold_group','isoform','y_true','y_pred','ci_low','ci_high']
assert set(required).issubset(preds),f'Missing OOF columns: {set(required)-set(preds)}'
assert set(preds.isoform)==set(ISOFORMS)
assert set(preds.fold)==set(range(5))
assert not preds.duplicated(['subset','canonical_smiles','isoform']).any()
assert np.isfinite(preds[['y_true','y_pred','ci_low','ci_high']].to_numpy(float)).all()
assert (preds.ci_low<=preds.ci_high).all()
assert manifest11.canonical_smiles.is_unique
assert manifest11.groupby('scaffold_group').fold.nunique().max()==1
assert preds.groupby(['canonical_smiles','isoform']).subset.nunique().eq(preds.subset.nunique()).all()
print('Validated 5 folds, 4 isoforms,',preds.canonical_smiles.nunique(),
      'molecules,',preds[['canonical_smiles','isoform']].drop_duplicates().shape[0],
      'labeled observations,',preds.subset.nunique(),'model configurations.')

## 4 · MA-ST-RAE results: raw OOF and 1,000 paired bootstrap resamples

The challenge bootstraps molecular identities, not the individual CYP labels. This is important because the public training table has missing measurements; for each sampled molecule, include only the CYP labels that actually exist. The same 1,000 index vectors are reused for every model to produce comparable paired model differences. The official blind leaderboard uses its own sealed test set, so these are **local OOF estimates**.

In [ ]:
def prepare_oof_matrices(frame, manifest, isoforms=ISOFORMS):
    ids=manifest.canonical_smiles.astype(str).tolist()
    structures=pd.Index(ids,name='canonical_smiles')
    model_matrices={}
    for name,group in frame.groupby('subset',sort=True):
        by_iso={}
        for iso in isoforms:
            si=group[group.isoform.eq(iso)].set_index('canonical_smiles')
            by_iso[iso]={column:si[column].reindex(structures).to_numpy(float)
                         for column in ['y_true','y_pred','ci_low','ci_high']}
        model_matrices[name]=by_iso
    return ids,model_matrices

molecule_ids,MATRICES=prepare_oof_matrices(preds,manifest11)
MODEL_NAMES=list(MATRICES)
N=len(molecule_ids)

def score_matrix(matrices, sampled_rows=None):
    if sampled_rows is None: sampled_rows=np.arange(N)
    per={}
    for iso in ISOFORMS:
        d=matrices[iso]; y=d['y_true'][sampled_rows]
        mask=np.isfinite(y)
        per[iso]=st_rae(y[mask],d['y_pred'][sampled_rows][mask],
                        d['ci_low'][sampled_rows][mask],d['ci_high'][sampled_rows][mask])
    return per

point=[]
for model in MODEL_NAMES:
    stats=score_matrix(MATRICES[model]);
    point.append({'subset':model,'MA_ST_RAE_single_pass':np.mean(list(stats.values())),
                  **{f'{k}_ST_RAE_single_pass':v for k,v in stats.items()}})
point_scores=pd.DataFrame(point).sort_values('MA_ST_RAE_single_pass')
point_scores.to_csv(OUT/'local_oof_point_scores.csv',index=False)
print('Single-pass OOF score (not leaderboard):')
display(point_scores.round(5))

rng=np.random.default_rng(BOOTSTRAP_SEED)
records=[]
for iteration in range(N_BOOTSTRAP):
    chosen=rng.integers(0,N,size=N)
    for model in MODEL_NAMES:
        scores=score_matrix(MATRICES[model],chosen)
        records.append({'replicate':iteration,'subset':model,'MA_ST_RAE':np.mean(list(scores.values())),
                        **{f'{k}_ST_RAE':v for k,v in scores.items()}})
    if (iteration+1)%250==0:print('Bootstrap',iteration+1,'/',N_BOOTSTRAP)
bootstrap=pd.DataFrame(records)
summary=[]
score_cols=['MA_ST_RAE']+[f'{iso}_ST_RAE' for iso in ISOFORMS]
for model,g in bootstrap.groupby('subset'):
    rec={'subset':model}
    for metric in score_cols:
        v=g[metric].to_numpy(float)
        rec[metric+'_mean']=v.mean()
        rec[metric+'_std']=v.std(ddof=0)
        rec[metric+'_ci_low']=np.quantile(v,.025)
        rec[metric+'_ci_high']=np.quantile(v,.975)
    summary.append(rec)
summary=pd.DataFrame(summary).sort_values('MA_ST_RAE_mean')
bootstrap.to_csv(OUT/'local_oof_bootstrap_replicates.csv',index=False)
summary.to_csv(OUT/'local_oof_bootstrap_summary.csv',index=False)
print('Local out-of-fold bootstrap scores (lower is better):')
display(summary[['subset','MA_ST_RAE_mean','MA_ST_RAE_std','MA_ST_RAE_ci_low','MA_ST_RAE_ci_high']].round(5))

## 5 · Paired differences and endpoint contributions

The paired comparison uses the same bootstrap resamples. These intervals condition on Notebook 11's fitted models, nested-weight selection, one scaffold partition, and the public training population; they are not uncertainty intervals for the unreleased test labels.

In [ ]:
PRIMARY_MODEL='CheMeleon_finetuned + ScreenStack_LGBM'
TRIPLE_MODEL='Descriptor_LGBM + CheMeleon_finetuned + ScreenStack_LGBM'
assert PRIMARY_MODEL in MODEL_NAMES and TRIPLE_MODEL in MODEL_NAMES
pair_rows=[]
for comparator in MODEL_NAMES:
    if comparator==PRIMARY_MODEL:continue
    b=bootstrap.pivot(index='replicate',columns='subset',values='MA_ST_RAE')
    diff=(b[comparator]-b[PRIMARY_MODEL]).to_numpy(float)
    pair_rows.append({'comparator':comparator,'candidate':PRIMARY_MODEL,
        'MA_ST_RAE_improvement_mean':diff.mean(),
        'bootstrap_ci_low':np.quantile(diff,.025),'bootstrap_ci_high':np.quantile(diff,.975)})
comparisons=pd.DataFrame(pair_rows).sort_values('MA_ST_RAE_improvement_mean',ascending=False)
comparisons.to_csv(OUT/'local_oof_paired_improvements.csv',index=False)
display(comparisons.round(5))

fig,axs=plt.subplots(1,2,figsize=(13,5))
q=summary.sort_values('MA_ST_RAE_mean')
axs[0].barh(q.subset.iloc[::-1],q.MA_ST_RAE_mean.iloc[::-1])
axs[0].set_xlabel('MA-ST-RAE (local OOF bootstrap mean; lower is better)')
s=q[q.subset==PRIMARY_MODEL].iloc[0]
axs[1].bar(ISOFORMS,[s[f'{iso}_ST_RAE_mean'] for iso in ISOFORMS])
axs[1].set_title('Selected two-model ensemble')
axs[1].set_ylabel('Endpoint ST-RAE')
axs[1].tick_params(axis='x',rotation=20)
fig.tight_layout();fig.savefig(OUT/'local_oof_official_metric_comparison.png',dpi=175)
plt.show()

## 6 · Download public files for **blind inference**

`RUN_BLIND_REFIT=True` triggers the final inference pipeline below. A **fresh refit** is essential: the OOF file contains *out-of-fold predictions for training molecules*, not predictions for the 750 blind test structures, and Notebook 11 did not export fitted model checkpoints. The final graph model is fine-tuned on the public training data (reserving an internal scaffold-held-out dev partition for early stopping); the screening first stage is trained on all eligible public screening data and cross-fitted to generate leakage-safe stage-two training features.

**Important:** test-set scaffold similarity to the public training set is expected in this analog-expansion challenge; this is *not* leakage. Exact test labels and screening measurements remain unavailable.

In [ ]:
def download_source(key):
    if key in DATA_OVERRIDES:
        blob=Path(DATA_OVERRIDES[key]).read_bytes()
    else:
        with urlopen(ROOT_URL+FILES[key],timeout=240) as fh:blob=fh.read()
    return pd.read_csv(io.BytesIO(blob)),hashlib.sha256(blob).hexdigest()

def canonical_scaffold(s):
    if not isinstance(s,str) or not s.strip():return None,None
    mol=Chem.MolFromSmiles(s)
    if mol is None:return None,None
    can=Chem.MolToSmiles(mol,canonical=True,isomericSmiles=True)
    core=MurckoScaffold.GetScaffoldForMol(mol)
    scaffold=Chem.MolToSmiles(core,canonical=True,isomericSmiles=False) if core.GetNumAtoms() else 'ACYCLIC_'+can
    return can,scaffold

if RUN_BLIND_REFIT:
    source={};data_sha={}
    for key in FILES:
        source[key],data_sha[key]=download_source(key)
        print('Downloaded',key,source[key].shape,'SHA256',data_sha[key][:16])
    primary=source['direct'].copy().reset_index(drop=True)
    blind=source['test'].copy().reset_index(drop=True)
    if not {'Molecule_Name','SMILES'}.issubset(blind):raise ValueError('Blind file must contain Molecule_Name and SMILES')
    assert len(blind)==750, 'The regression blind set must contain exactly 750 molecules'
    assert blind.Molecule_Name.notna().all() and blind.Molecule_Name.is_unique
    assert not set(f'{iso}_pIC50_direct_inhibition' for iso in ISOFORMS).intersection(blind.columns),(
        'Unexpected target labels in blind file; abort to prevent test-label leakage')
    direct_ident=[canonical_scaffold(x) for x in primary.SMILES]
    primary['canonical_smiles']=[a for a,b in direct_ident]
    assert primary.canonical_smiles.notna().all() and primary.canonical_smiles.is_unique
    blind_ident=[canonical_scaffold(x) for x in blind.SMILES]
    blind['canonical_smiles']=[a for a,b in blind_ident]
    assert blind.canonical_smiles.notna().all()
    assert blind.canonical_smiles.is_unique, 'Duplicate canonical SMILES in blind test; audit source before proceeding'
    if set(blind.canonical_smiles)&set(primary.canonical_smiles):
        print('NOTE: exact SMILES overlap exists; predictions still use no test labels')
    for iso in ISOFORMS:assert f'{iso}_pIC50_direct_inhibition' in primary
    # Single-concentration response is an auxiliary training label only.
    screen=source['single'].copy()
    for req in ['SMILES','enzyme','log2fc_estimate']:
        assert req in screen, f'Screening file missing {req}'
    screen['canonical_smiles']=[canonical_scaffold(x)[0] for x in screen.SMILES]
    screen['screen_value']=pd.to_numeric(screen.log2fc_estimate,errors='coerce')
    screen['enzyme']=screen.enzyme.astype(str).str.strip().str.upper()
    screen=screen.dropna(subset=['canonical_smiles','screen_value'])
    screen=screen[screen.enzyme.isin(ISOFORMS)]
    aggregated=screen.groupby(['canonical_smiles','enzyme'],as_index=False).screen_value.mean()
    all_smiles=list(primary.canonical_smiles)
    extra=sorted(set(aggregated.canonical_smiles)-set(all_smiles))
    all_smiles+=extra
    N_PRIMARY=len(primary); N_TRAIN_STRUCTURES=len(all_smiles)
    BLIND_START=N_TRAIN_STRUCTURES
    all_smiles+=list(blind.canonical_smiles)
    all_groups=np.asarray([canonical_scaffold(s)[1] for s in all_smiles],object)
    N_ALL=len(all_smiles)
    # The first N_PRIMARY indices are labeled primary training compounds.
    Y=np.column_stack([pd.to_numeric(primary[f'{iso}_pIC50_direct_inhibition'],errors='coerce').to_numpy(float)
                       for iso in ISOFORMS])
    SCREEN_Y=np.full((N_ALL,4),np.nan)
    all_index={s:i for i,s in enumerate(all_smiles[:N_TRAIN_STRUCTURES])}
    for row in aggregated.itertuples(index=False):
        SCREEN_Y[all_index[row.canonical_smiles],ISOFORMS.index(row.enzyme)]=row.screen_value
    assert np.isfinite(SCREEN_Y[:N_TRAIN_STRUCTURES]).sum()>0
    assert np.isnan(SCREEN_Y[BLIND_START:]).all(), 'Do not load test screening measurements!'
    TRAIN_IDS=np.arange(N_PRIMARY,dtype=int)
    BLIND_IDS=np.arange(BLIND_START,N_ALL,dtype=int)
    assert len(BLIND_IDS)==750
    print('Labeled primary:',N_PRIMARY,'auxiliary-only:',len(extra),'unlabeled blind:',len(BLIND_IDS))
else:
    print('Skipped blind refit: local OOF scoring only.')

## 7 · Reproduce Notebook 11 molecular features and model definitions

The code below uses the same ECFP4 count fingerprints, RDKit descriptors, cross-fitted screening regression, and pretrained CheMeleon fine-tuning as Notebook 11. The `guard_groups` function **does not demand scaffold separation between public training data and the external blind test set** (because the challenge deliberately uses analog expansion), but does enforce scaffold separation for internal cross-fitting and internal graph early stopping.

In [ ]:
if RUN_BLIND_REFIT:
    feature_key=hashlib.sha256(json.dumps({'sha':data_sha,'fp_bits':FINGERPRINT_BITS,
        'rdkit':rdBase.rdkitVersion,'descriptors':[n for n,fn in Descriptors._descList]},
        sort_keys=True).encode()).hexdigest()[:16]
    feature_file=CACHE/f'blind_features_{feature_key}.npz'
    if feature_file.exists():
        with np.load(feature_file) as z:X_RAW=z['X_RAW']
        print('Loaded cached molecular features:',X_RAW.shape)
    else:
        gen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=FINGERPRINT_BITS,countSimulation=False)
        desc_fns=[fn for _,fn in Descriptors._descList]
        X_RAW=np.empty((N_ALL,FINGERPRINT_BITS+len(desc_fns)),np.float32)
        for i,smi in enumerate(all_smiles):
            mol=Chem.MolFromSmiles(smi)
            X_RAW[i,:FINGERPRINT_BITS]=np.asarray(gen.GetCountFingerprintAsNumPy(mol),np.float32)
            for j,fn in enumerate(desc_fns):
                try:X_RAW[i,FINGERPRINT_BITS+j]=float(fn(mol))
                except Exception:X_RAW[i,FINGERPRINT_BITS+j]=np.nan
            if (i+1)%1000==0:print('Featurized',i+1,'/',N_ALL)
        X_RAW[~np.isfinite(X_RAW)]=np.nan
        np.savez_compressed(feature_file,X_RAW=X_RAW)
    assert X_RAW.shape[0]==N_ALL
    print('Feature matrix:',X_RAW.shape)

In [ ]:
if RUN_BLIND_REFIT:
    import torch
    from sklearn.model_selection import GroupKFold as _GroupKFold
    from lightning import pytorch as pl
    from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

    torch.manual_seed(SEED);pl.seed_everything(SEED,workers=True)
    DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
    print('Torch:',torch.__version__,'GPU available:',torch.cuda.is_available())

    if RUN_GRAPH and not GRAPH_PROXY_SMOKE_ONLY:
        from chemprop import featurizers, nn as cnn, data as cdata, models as cmodels
        import chemprop
        ZENODO='https://zenodo.org/records/15460715/files/chemeleon_mp.pt'
        CKPT=CACHE/'chemeleon_mp.pt'
        if not CKPT.exists():
            print('Downloading original CheMeleon message-passing weights from Zenodo')
            urlretrieve(ZENODO,CKPT)
        assert CKPT.stat().st_size>100000,'Unexpectedly small CheMeleon checkpoint'
        CKPT_SHA=hashlib.sha256(CKPT.read_bytes()).hexdigest()
        print('Pretrained checkpoint SHA256:',CKPT_SHA)
    else:
        CKPT_SHA='PROXY_ONLY' if GRAPH_PROXY_SMOKE_ONLY else 'DISABLED'

In [ ]:
if RUN_BLIND_REFIT:
    SETTINGS={'tag':EXPERIMENT_TAG,'data_sha256':data_sha,'seed':SEED,
        'fingerprint_bits':FINGERPRINT_BITS,'screen_crossfit_folds':SCREEN_CROSSFIT_FOLDS,
        'lgbm_estimators':N_LGBM_ESTIMATORS,'screen_estimators':N_SCREEN_ESTIMATORS,
        'graph_max_epochs':MAX_GRAPH_EPOCHS,'graph_patience':GRAPH_PATIENCE,
        'checkpoint_sha256':CKPT_SHA,'graph_proxy_smoke_only':GRAPH_PROXY_SMOKE_ONLY,
        'graph_enabled':RUN_GRAPH,'fast_mode':FAST_MODE,'lightgbm':lgb.__version__,
        'rdkit':rdBase.rdkitVersion,'torch':torch.__version__}
    SIGNATURE=hashlib.sha256(json.dumps(SETTINGS,sort_keys=True).encode()).hexdigest()[:20]
    RUN_CACHE=CACHE/SIGNATURE;RUN_CACHE.mkdir(parents=True,exist_ok=True)
    print('Final-fit cache signature:',SIGNATURE)

In [ ]:
if RUN_BLIND_REFIT:
    AUDIT_LOG=[]
    def guard_groups(train_idx,predict_idx,outer_holdout_idx,what):
        train_groups=set(all_groups[np.asarray(train_idx,dtype=int)])
        held_idx=np.asarray(outer_holdout_idx,dtype=int)
        held_groups=set() if len(held_idx) and np.all(held_idx>=N_TRAIN_STRUCTURES) else set(all_groups[held_idx])
        pred_groups=set(all_groups[np.asarray(predict_idx,dtype=int)])
        external=bool(len(predict_idx)) and np.all(np.asarray(predict_idx,dtype=int)>=N_TRAIN_STRUCTURES)
        overlap_outer=len(train_groups&held_groups)
        overlap_pred=len(train_groups&pred_groups)
        if external:
            assert not set(np.asarray(train_idx,dtype=int)) & set(np.asarray(predict_idx,dtype=int)), 'Exact row leakage'
            overlap_outer=0;overlap_pred=0  # External unlabeled analogs may share scaffolds.
        else:
            assert overlap_outer==0 and overlap_pred==0, f'Leakage in {what}: outer={overlap_outer}, validation={overlap_pred}'
        AUDIT_LOG.append(dict(step=what,n_fit=len(train_idx),n_pred=len(predict_idx),
            outer_scaffold_overlap=overlap_outer,validation_scaffold_overlap=overlap_pred))

    def impute_for(train_idx,pred_idx):
        imp=SimpleImputer(strategy='median',keep_empty_features=True)
        fit=imp.fit_transform(X_RAW[train_idx]).astype(np.float32)
        val=imp.transform(X_RAW[pred_idx]).astype(np.float32)
        assert fit.shape[1]==val.shape[1]
        return fit,val

    def named_fit_predict(model,x_fit,y_fit,x_eval):
        # LightGBM's sklearn wrapper retains DataFrame feature names; provide matching names at prediction.
        cols=[f'f_{k:05d}' for k in range(x_fit.shape[1])]
        model.fit(pd.DataFrame(x_fit,columns=cols),y_fit)
        return model.predict(pd.DataFrame(x_eval,columns=cols))

    def lgbm_model(seed,stage='potency'):
        return lgb.LGBMRegressor(n_estimators=N_SCREEN_ESTIMATORS if stage=='screen' else N_LGBM_ESTIMATORS,
            learning_rate=.04,num_leaves=15,min_child_samples=6 if FAST_MODE else 15,
            colsample_bytree=.8,subsample=.85,subsample_freq=1,
            reg_lambda=3.,verbosity=-1,n_jobs=2,random_state=seed)

    def fit_direct(train_idx,pred_idx,outer_idx,seed):
        guard_groups(train_idx,pred_idx,outer_idx,'direct_LGBM')
        a,b=impute_for(train_idx,pred_idx)
        result=np.full((len(pred_idx),4),np.nan,float)
        for j in range(4):
            selected=np.isfinite(Y[train_idx,j])
            if selected.sum()<6: raise ValueError('Insufficient direct labels for '+ISOFORMS[j])
            if np.std(Y[train_idx[selected],j])<1e-12:
                result[:,j]=float(np.mean(Y[train_idx[selected],j]));continue
            reg=lgbm_model(seed+j)
            result[:,j]=named_fit_predict(reg,a[selected],Y[train_idx[selected],j],b)
        return result

    def predict_screen_from_fit(screen_fit_idx,query_idx,seed,why,outer_idx):
        guard_groups(screen_fit_idx,query_idx,outer_idx,why)
        x_fit,x_q=impute_for(screen_fit_idx,query_idx)
        result=np.full((len(query_idx),4),np.nan,float)
        for j in range(4):
            observed=np.isfinite(SCREEN_Y[screen_fit_idx,j])
            if observed.sum()<6 or np.std(SCREEN_Y[screen_fit_idx[observed],j])<1e-10:
                val=SCREEN_Y[screen_fit_idx[observed],j]
                result[:,j]=float(np.mean(val)) if len(val) else 0.;continue
            model=lgbm_model(seed+j,stage='screen')
            result[:,j]=named_fit_predict(model,x_fit[observed],SCREEN_Y[screen_fit_idx[observed],j],x_q)
        assert np.isfinite(result).all()
        return result

    def fit_screenstack(train_idx,pred_idx,outer_idx,seed):
        # train_idx and pred_idx contain PRIMARY row indices; aux-only structures may be
        # admitted to FIRST-STAGE screen training iff their scaffold is eligible.
        guard_groups(train_idx,pred_idx,outer_idx,'screen_stack_second_stage')
        external=bool(len(pred_idx)) and np.all(np.asarray(pred_idx,dtype=int)>=N_TRAIN_STRUCTURES)
        forbidden=set() if external else (set(all_groups[outer_idx])|set(all_groups[pred_idx]))
        eligible=np.asarray([i for i,g in enumerate(all_groups[:N_TRAIN_STRUCTURES]) if g not in forbidden],int)
        t_groups=all_groups[train_idx]
        n_inner=min(SCREEN_CROSSFIT_FOLDS,len(set(t_groups)))
        if n_inner<2: raise ValueError('Need >=2 distinct training scaffolds for cross-fitted screening features')
        train_screen=np.full((len(train_idx),4),np.nan,float)
        for sf,(fit_pos,val_pos) in enumerate(_GroupKFold(n_splits=n_inner).split(train_idx,groups=t_groups)):
            cf_valid=train_idx[val_pos]
            val_groups=set(all_groups[cf_valid])
            sf_fit=np.asarray([k for k in eligible if all_groups[k] not in val_groups],int)
            train_screen[val_pos]=predict_screen_from_fit(sf_fit,cf_valid,seed+100*sf,
                f'screen_stage1_crossfit_{sf}',outer_idx)
        assert np.isfinite(train_screen).all()
        pred_screen=predict_screen_from_fit(eligible,pred_idx,seed+99999,
            'screen_stage1_eval',outer_idx)
        x_tr,x_eval=impute_for(train_idx,pred_idx)
        x_tr=np.column_stack([x_tr,train_screen]).astype(np.float32)
        x_eval=np.column_stack([x_eval,pred_screen]).astype(np.float32)
        result=np.full((len(pred_idx),4),np.nan,float)
        for j in range(4):
            observed=np.isfinite(Y[train_idx,j])
            if observed.sum()<6:raise ValueError('Too few direct labels for screen stack '+ISOFORMS[j])
            reg=lgbm_model(seed+2000+j)
            result[:,j]=named_fit_predict(reg,x_tr[observed],Y[train_idx[observed],j],x_eval)
        return result

    def train_chemeleon(train_idx,pred_idx,outer_idx,seed,workdir):
        guard_groups(train_idx,pred_idx,outer_idx,'chemeleon_training')
        if GRAPH_PROXY_SMOKE_ONLY:
            # Proves folds/weight selection work without a checkpoint; NOT CheMeleon.
            pred=fit_direct(train_idx,pred_idx,outer_idx,seed+777)
            return pred
        if not RUN_GRAPH: raise RuntimeError('RUN_GRAPH must be True for three-model experiment')
        if DEVICE!='cuda':warnings.warn('Fine-tuning CheMeleon without GPU may be very slow')
        unique_groups=len(set(all_groups[train_idx]))
        if unique_groups<3:raise ValueError('Insufficient scaffolds for internal graph dev split')
        fit_pos,dev_pos=next(GroupShuffleSplit(n_splits=1,test_size=.15,random_state=seed).split(
            train_idx,groups=all_groups[train_idx]))
        fit_idx=train_idx[fit_pos];dev_idx=train_idx[dev_pos]
        guard_groups(fit_idx,dev_idx,outer_idx,'chemeleon_earlystop_dev')
        def point(i,labels):
            return cdata.MoleculeDatapoint.from_smi(all_smiles[i],np.asarray(labels,dtype=np.float32))
        featurizer=featurizers.SimpleMoleculeMolGraphFeaturizer()
        ds_train=cdata.MoleculeDataset([point(i,Y[i]) for i in fit_idx],featurizer)
        scaler=ds_train.normalize_targets()  # fit scaler only on fitting molecules
        ds_val=cdata.MoleculeDataset([point(i,Y[i]) for i in dev_idx],featurizer)
        ds_val.normalize_targets(scaler)
        ds_test=cdata.MoleculeDataset([cdata.MoleculeDatapoint.from_smi(all_smiles[i]) for i in pred_idx],featurizer)
        loaders=[cdata.build_dataloader(ds,batch_size=GRAPH_BATCH_SIZE,num_workers=0,shuffle=shuffle)
                 for ds,shuffle in [(ds_train,True),(ds_val,False),(ds_test,False)]]
        checkpoint_obj=torch.load(CKPT,map_location='cpu',weights_only=True)
        mp=cnn.BondMessagePassing(**checkpoint_obj['hyper_parameters'])
        mp.load_state_dict(checkpoint_obj['state_dict'])
        predictor=cnn.RegressionFFN(n_tasks=4,input_dim=mp.output_dim,
            output_transform=cnn.UnscaleTransform.from_standard_scaler(scaler))
        net=cmodels.MPNN(mp,cnn.MeanAggregation(),predictor,batch_norm=False,
            metrics=[cnn.metrics.MAE()])
        ckpt_dir=Path(workdir)/'trusted_local_checkpoints';ckpt_dir.mkdir(parents=True,exist_ok=True)
        checkpoint=ModelCheckpoint(dirpath=ckpt_dir,filename='best',monitor='val_loss',mode='min',save_top_k=1)
        stopper=EarlyStopping(monitor='val_loss',mode='min',patience=GRAPH_PATIENCE)
        trainer=pl.Trainer(max_epochs=MAX_GRAPH_EPOCHS,
            accelerator='gpu' if torch.cuda.is_available() else 'cpu',devices=1,
            logger=False,enable_model_summary=False,enable_progress_bar=False,
            callbacks=[checkpoint,stopper],gradient_clip_val=1.0,log_every_n_steps=10)
        torch.manual_seed(seed)
        trainer.fit(net,loaders[0],loaders[1])
        assert checkpoint.best_model_path,'CheMeleon training produced no best checkpoint'
        with torch.inference_mode():
            batches=trainer.predict(net,dataloaders=loaders[2],ckpt_path='best',weights_only=False)
        pred=np.concatenate([np.asarray(t) for t in batches],axis=0).reshape(len(pred_idx),4).astype(float)
        del trainer,net,mp
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        return pred

## 8 · Final full-public-train refit and blind inference (restartable)

This performs one CheMeleon fine-tuning run and one cross-fitted screening-to-potency LightGBM training run, both trained exclusively on the public training data. Results are stored with a hash of the input files and model parameters. The two predictions are blended with the **preselected fixed 50:50 weights**.

A refit-based blind-test submission is not the same fit as a held-out-fold model; its performance must be estimated by the OOF study above, never by fitting against unknown blind labels.

In [ ]:
if RUN_BLIND_REFIT:
    audit_path=RUN_CACHE/'blind_fit_audit.csv'
    pred_path=RUN_CACHE/'blind_component_predictions.npz'
    AUDIT_LOG=[]
    if pred_path.exists():
        with np.load(pred_path) as z:
            chemeleon_pred=z['CheMeleon_finetuned']
            screening_pred=z['ScreenStack_LGBM']
        print('Loaded cached full-public-train blind model predictions.')
        if audit_path.exists(): AUDIT_LOG=pd.read_csv(audit_path).to_dict('records')
    else:
        screen_pred=fit_screenstack(TRAIN_IDS,BLIND_IDS,BLIND_IDS,SEED+101)
        chemeleon_pred=train_chemeleon(TRAIN_IDS,BLIND_IDS,BLIND_IDS,
                                      SEED+202,RUN_CACHE/'chemeleon_full_train')
        screening_pred=screen_pred
        assert chemeleon_pred.shape==screening_pred.shape==(len(BLIND_IDS),4)
        assert np.isfinite(chemeleon_pred).all() and np.isfinite(screening_pred).all()
        # Atomic cache write; do not reuse an incomplete file after disconnection.
        tmp=pred_path.with_name('temp_component_predictions.npz')
        np.savez_compressed(tmp,CheMeleon_finetuned=chemeleon_pred,ScreenStack_LGBM=screening_pred)
        os.replace(tmp,pred_path)
        pd.DataFrame(AUDIT_LOG).to_csv(audit_path,index=False)
        print('Saved cached blind predictions to',pred_path)
    final_pred=.5*chemeleon_pred+.5*screening_pred
    assert final_pred.shape==(750,4) and np.isfinite(final_pred).all()
    print('Final blend: 50% fine-tuned CheMeleon + 50% screen-stacked LightGBM')
    display(pd.DataFrame(final_pred,columns=ISOFORMS).describe().round(3))

## 9 · Strict 750-row submission validation

**Only the direct-inhibition regression track** is covered. This notebook does **not** produce the separate CYP2D6/CYP3A4 TDI classification entry.

The final CSV uses the organizers' unmodified `Molecule_Name` identifiers and four exact direct-inhibition endpoint column names, one prediction per test molecule, preserving both original `Molecule_Name` and `SMILES`. An ID-only alternate CSV is also exported if the portal only accepts the five-column layout. An audit checks IDs, row count, finiteness, and correspondence to the original blinded CSV. No experimental test values are included. Full/fast/proxy status is recorded in the metadata.

In [ ]:
def validate_regression_submission(submission,blind_ids,allow_synthetic=False):
    columns=['Molecule_Name','SMILES']+[f'{iso}_pIC50_direct_inhibition' for iso in ISOFORMS]
    assert list(submission.columns)==columns, f'Wrong submission columns; expected {columns}'
    assert len(submission)==750 and len(blind_ids)==750
    assert submission.Molecule_Name.notna().all() and submission.Molecule_Name.is_unique
    assert list(submission.Molecule_Name.astype(str))==list(pd.Series(blind_ids).astype(str)),(
        'Submission IDs or order differ from official blind test file')
    assert submission.SMILES.notna().all() and submission.SMILES.astype(str).str.len().gt(0).all()
    values=submission[columns[2:]].to_numpy(float)
    assert values.shape==(750,4) and np.isfinite(values).all()
    assert not submission.duplicated('Molecule_Name').any()
    return {'n_compounds':len(submission),'n_predictions':values.size,
            'columns':columns,'min':float(values.min()),'max':float(values.max()),
            'mean':float(values.mean())}

if RUN_BLIND_REFIT:
    regression_cols=[f'{iso}_pIC50_direct_inhibition' for iso in ISOFORMS]
    submission=pd.DataFrame(final_pred,columns=regression_cols)
    submission.insert(0,'SMILES',blind.SMILES.to_numpy(copy=True))
    submission.insert(0,'Molecule_Name',blind.Molecule_Name.to_numpy(copy=True))
    audit=validate_regression_submission(submission,blind.Molecule_Name)
    assert np.array_equal(submission.SMILES.to_numpy(),blind.SMILES.to_numpy())
    scientific_submission=bool(not FAST_MODE and not GRAPH_PROXY_SMOKE_ONLY and RUN_GRAPH)
    filename='direct_inhibition_submission.csv' if scientific_submission else 'SMOKE_ONLY_NOT_FOR_SUBMISSION.csv'
    output_submission=OUT/filename
    submission.to_csv(output_submission,index=False,float_format='%.8f')
    # Re-read what we saved; verify CSV round-trip and exact identity.
    reread=pd.read_csv(output_submission)
    validate_regression_submission(reread,blind.Molecule_Name)
    assert np.array_equal(reread.SMILES.to_numpy(),blind.SMILES.to_numpy())
    minimal=submission.drop(columns='SMILES')
    minimal.to_csv(OUT/('direct_inhibition_submission_ID_only.csv' if scientific_submission else 'SMOKE_ONLY_ID_only.csv'),index=False,float_format='%.8f')
    print('Submission-format audit:',json.dumps(audit,indent=2))
    print('Wrote',output_submission,'|', 'REAL FIT: format only' if scientific_submission else 'SMOKE ONLY: DO NOT UPLOAD')
    assert 'ST_RAE' not in filename, 'Blind test is unscored: do not label its predictions as scored.'
    component_out=pd.DataFrame({'Molecule_Name':blind.Molecule_Name})
    for j,iso in enumerate(ISOFORMS):
        component_out[f'{iso}_CheMeleon']=chemeleon_pred[:,j]
        component_out[f'{iso}_ScreenStack']=screening_pred[:,j]
    component_out.to_csv(OUT/'blind_unscored_component_predictions.csv',index=False)
else:
    print('No blind inference requested. The local OOF metric reports are complete.')

## 10 · Provenance, machine-readable reports, and download ZIP

This ZIP includes the 1,000-resample local OOF score distribution, per-isoform scores, paired comparisons, image, source-code parity result, run settings, and—only if `RUN_BLIND_REFIT`—the **unscored** test predictions. Keep the source training files and pretrained checkpoint provenance with your GitHub project.

A successful format audit does **not** mean the submission was accepted by the challenge platform. Upload the CSV yourself via the official CYP direct-inhibition regression submission form. Do not submit synthetic or proxy predictions.

In [ ]:
metadata={'notebook':'12_ma_st_rae_blind_evaluation','evaluation_type':'PUBLIC_TRAIN_OUTER_OOF',
    'cannot_score_blind_test_without_ground_truth':True,
    'n_bootstrap':N_BOOTSTRAP,'bootstrap_seed':BOOTSTRAP_SEED,
    'reference_scorer_parity':OFFICIAL_PARITY,'nb11_zip_sha256':hashlib.sha256(nb11_path.read_bytes()).hexdigest(),
    'n_primary_scored_molecules':len(molecule_ids),'isof':ISOFORMS,
    'selected_model':PRIMARY_MODEL,'selected_fixed_weights':{'CheMeleon_finetuned':0.5,'ScreenStack_LGBM':0.5},
    'blind_refit_run':RUN_BLIND_REFIT,'fast_mode':FAST_MODE,'graph_proxy_smoke_only':GRAPH_PROXY_SMOKE_ONLY,
    'description':'Test SMILES are blind: no score calculated; official test MA-ST-RAE requires organizers labels'}
if RUN_BLIND_REFIT:
    metadata.update({'data_shas':data_sha,'run_signature':SIGNATURE,
        'blind_output_filename':filename,'n_blind':len(blind),
        'fit_model_settings':SETTINGS})
    if AUDIT_LOG:
        pd.DataFrame(AUDIT_LOG).to_csv(OUT/'blind_fit_leakage_and_provenance_audit.csv',index=False)
(OUT/'run_metadata.json').write_text(json.dumps(metadata,indent=2,default=str))
archive_path=BASE/'cyp_notebook_12_results.zip'
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file():z.write(f,OUT.name+'/'+f.name)
print('Saved results archive:',archive_path.resolve(),
      'size MiB:',round(archive_path.stat().st_size/1048576,2))
print('Files:',*[x.name for x in OUT.iterdir() if x.is_file()],sep='\n - ')
print('Next step: submit only a valid direct_inhibition_submission.csv, if produced by REAL full fit.')

## 11 · Interpretation checklist

- [ ] `OFFICIAL_PARITY` passed or any formula mismatch is investigated against the upstream scorer.
- [ ] OOF source metadata confirms all five real CheMeleon outer folds; no fast/proxy artifacts.
- [ ] Four endpoint intervals are finite and correctly paired to the `y_true` values.
- [ ] Report **bootstrapped MA-ST-RAE** and per-endpoint ST-RAE, not just overall MAE or interval excess.
- [ ] Distinguish full-public-training refit from the nested CV fits used for OOF evaluation.
- [ ] Use `FAST_MODE=False`, `GRAPH_PROXY_SMOKE_ONLY=False` for any real submission.
- [ ] Confirm `Molecule_Name` identities, 750 rows and four finite pIC50 columns, then **upload manually** to the competition site.
- [ ] The **blind-test MA-ST-RAE remains unknown** until the organizer scores the submission.

**Scientific caveat:** Training-set OOF macro ST-RAE may differ materially from test performance because the test compounds were selected as potent-hit analogs rather than as a random subset of the primary training chemical space. The OOF analysis is useful for ranking development methods but is not the leaderboard score.